# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [35]:
import pandas as pd
import numpy as np
import requests
from pathlib import *
import os
import sys
from dotenv import load_dotenv
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error
import polars as pl
from tqdm import tqdm
from typing import Union
import csv
import re
import matplotlib.pyplot as plt
import random

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)


from utils import model_connector
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, AIMessage, SystemMessage


# Data preprocess

In [3]:
# Step 1: Load Data
def data_loader(file_path: str) -> pd.DataFrame:
    data = pd.read_csv(file_path)
    data = data.rename(columns={"Date": "date", "Open": "open", "Close": "close", "High": "high", "Low": "low" })
    data.set_index("date", inplace=True)
    return data

# Step 1.1 - Resample to lower timeframe, Ignore If Daily
def data_resampler(df: pd.DataFrame, resample_timeframe: str) -> pd.DataFrame:
    df = df.resample("15min").agg({
        "open":            "first",
        "high":            "max",
        "low":             "min",
        "close":           "last",
        "volume":          "sum",
        "quote_volume":    "sum",
        "trades":          "sum",
        "taker_buy_base":  "sum",
        "taker_buy_quote": "sum",
    }).dropna()
    return df

# Step 1.2 - We split into train and test. Since we use windowing and former values to calculate the RV and returns we don't want leakage from 
# the train data into the test data. Best approach is to split the dataset first, then calculate the features, THEN drop the na's.
def get_train_test_df(data: pd.DataFrame, split_ratio: float = 0.6) -> Union[pd.DataFrame, pd.DataFrame]:
    train_data = data[:int(len(data)*split_ratio)]
    test_data = data[len(train_data): ]
    return train_data, test_data

# Write Functions to create features for both train and test. 
# Step 2: Calculate Log Returns
def log_returns(data: pd.DataFrame) -> pd.DataFrame:
    data["r"] = (data["close"]/data["close"].shift(1)) - 1 
    data["log_r"] = np.log(data["close"]/data["close"].shift(1))
    return data

# Step 3: Calculate Volatility (Square of log returns)
def volatility(data: pd.DataFrame) -> pd.DataFrame:
    data["rv"] = data["log_r"]**2
    return data

def rescale(data: pd.DataFrame) -> pd.DataFrame:
    data["log_r_scaled"] = data["log_r"] * 1e3
    data["rv_scaled"] = np.log(data["rv"])
    return data

# Step 4: Collect X and Y
def features(data: pd.DataFrame, window: int, strides: int, require_rescale: bool = False) -> Union[list, list]:
    if require_rescale:
        data = data.pipe(rescale)
        sample_data = data[["log_r_scaled", "rv_scaled"]].dropna().to_numpy()
    else:
        sample_data = data[["log_r", "rv"]].dropna().to_numpy()

    w = window 
    x = [] 
    y = []

    for i in tqdm(range(0, len(sample_data)-w, strides)):
        x.append(tuple(sample_data[i:i+w]))
        
    y = [sample_data[i+w][1] for i in range(0, len(sample_data)-w, strides)]

    return x, y

In [4]:
train_data, test_data = data_loader(file_path="../data/sp500.csv").pipe(get_train_test_df, split_ratio=0.6)

In [5]:
window = 7
strides = 1

# Train Prepare
x_train, y_train = train_data.pipe(log_returns).pipe(volatility).pipe(features, window=window, strides=strides)
# Test Prepare
x_test, y_test = test_data.pipe(log_returns).pipe(volatility).pipe(features, window=window, strides=strides)

print(f"Len of x_train: {len(x_train)}, Len of y_train: {len(y_train)}")
print(f"Len of x_test: {len(x_test)}, Len of y_test: {len(y_test)}")

100%|██████████| 1363/1363 [00:00<00:00, 202487.74it/s]

Len of x_train: 2048, Len of y_train: 2048
Len of x_test: 1363, Len of y_test: 1363


# 1. Connect model and get Initial Predictions

In [6]:
def formulate_p_input(x: tuple, p_input_prompt: str) -> HumanMessage:
    return HumanMessage(f"{x}\n{p_input_prompt}")
    
def formulate_p_query(p_query_prompt: str) -> SystemMessage:
    return SystemMessage(p_query_prompt)

In [7]:
p_input_prompt = """You are given a chronological sequence of observations for a financial asset.
Each row reports the 1-period log return log_r and the realized variance rv 
where rv = log_r^2.
It's arranged chronologically, [array([log_r_1, rv_1]), array([log_r_2, rv_2]), .....]
"""

p_query_prompt = """Task: Forecast realized variance at the next time step, t+1 from the 
given chronological dataset.

The forecast target is one step ahead of the final observation above,
reported in the same units as the sequence.

Respond with a single number. Output nothing else — no explanation, no units, no surrounding text.

You must output a single valid rv. It must be convertible to a float. Some inputs are very small. 

rv =
"""

### parse_float_response is completely AI generated
def parse_float_response(text: str) -> float:
    if not isinstance(text, str):
        return np.nan
    
    # Regex pattern to match floats, ints, and scientific notation (e.g. 3.73e-06)
    pattern = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"
    matches = re.findall(pattern, text)
    
    if not matches:
        return np.nan
    
    # Handle array([log_r, rv]) responses -> pick the last number (rv)
    if "array" in text or len(matches) > 1 and "[" in text:
        return float(matches[-1])
    
    # Default: take the first matched valid number (e.g., '0.000202:0' -> '0.000202')
    try:
        return float(matches[0])
    except ValueError:
        return np.nan


def get_initial_predictions(model, p_query_prompt: str, p_input_prompt: str):
    for i in tqdm(range(len(x_train))):
        # if i == 5:
        #     break
        p_input = formulate_p_input(x_train[i], p_input_prompt = p_input_prompt)
        p_query = formulate_p_query(p_query_prompt=p_query_prompt)
        p_conc = str(p_input.content)
        response = model.invoke([p_query, p_input])
        yield p_conc, parse_float_response(response.content)

In [ ]:
model: ChatOpenAI = model_connector.get_model(model_name="OpenAI")
with open('../outputs/outputs_initial.csv', 'w', newline='') as model_output:
    writer = csv.writer(model_output)
    writer.writerow(["p_conc", "initial_prediction"])
    for p_conc, pred in get_initial_predictions(model, p_query_prompt=p_query_prompt, p_input_prompt=p_input_prompt):
        writer.writerow([p_conc, pred])
        model_output.flush()

100%|██████████| 2048/2048 [28:37<00:00,  1.19it/s]


# 2. Oracle Guided Refinement

In [8]:
def model_prompt_for_refinement(y_pred, feedback: tuple) -> HumanMessage:
    y_true, _, abs_error, hint = feedback
    
    prompt = f"""Task: predict the realized variance of the NEXT trading day.
                You previously answered this task. Oracle feedback on your answer:
                Ground-truth realized variance (next day): {y_true}
                Your current prediction: {y_pred}
                Absolute error: {abs_error}
                {hint}
                Provide a revised prediction. Respond with a single number only."""    
    return HumanMessage(content=prompt)

def get_oracle_predictions(model, prompt):
        response = model.invoke([prompt])
        return parse_float_response(response.content)

def mae(y_true: np.array, y_pred: np.array):
    return np.abs(np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float))

def get_heuristic(index, y_true, y_pred, period) -> str:
    recent_avg = np.mean(y_true[index-period: index])
    return (f"Hint: your prediction is not the true realized variance. "
            f"The average realized variance over the last {period} days is {recent_avg:.4f}. "
            f"Volatility clusters, so next-day variance typically stays close to this "
            f"recent average; reason and revise your estimate.")

def feedback_func(index, y_true, y_pred, error_func: str="MAE", heuristic_period: int = 3):
    pred_error = None
    if error_func == "MAE":
        pred_error = mae(y_true, y_pred)
    # return pred_error
    preds = []
    model: ChatOpenAI = model_connector.get_model(model_name="OpenAI")
    with open(f'../outputs/outputs_{index}.csv', 'w', newline='') as model_output:
        writer = csv.writer(model_output)
        writer.writerow([f"refined_prediction_{index}"])
        for i in tqdm(range(len(y_true))):
            if i < heuristic_period:
                continue
            feedback = (y_true[i], y_pred[i], pred_error[i], get_heuristic(i, y_true, y_pred, heuristic_period))
            prompt = model_prompt_for_refinement(y_pred[i], feedback)
            output = get_oracle_predictions(model, prompt)
            writer.writerow([output])
            model_output.flush()
            preds.append(output)
    return preds


In [9]:
def training(refinement_loops: int = 3):
    y_pred = pd.read_csv("../outputs/outputs_initial.csv")["initial_prediction"].to_numpy()
    preds_iters = []
    heuristic_period = 3
    y_train_copy = y_train.copy()
    for i in range(0, refinement_loops):
        print(f"Refinement Loop: {i+1}")
        preds_iters.append(feedback_func(i, y_train_copy, y_pred, heuristic_period=heuristic_period))
        y_train_copy = y_train_copy[heuristic_period: ]
        y_pred = preds_iters[-1]
    return preds_iters

In [ ]:
refinement_loops: int = 3
refined = training(refinement_loops)

In [20]:
refinements = []
max_len = -1e9
for i in range(0, refinement_loops+1):
    if i==0:
        current_refinement = pd.read_csv("../outputs/outputs_initial.csv")        
    else:
        current_refinement = pd.read_csv(f"../outputs/outputs_{i-1}.csv")
    max_len = len(current_refinement) if len(current_refinement) > max_len else max_len
    current_refinement = current_refinement.reset_index(drop=True)
    current_refinement.index = range(max_len - len(current_refinement), max_len)
    refinements.append(current_refinement)


full_df = pd.concat(refinements, axis=1).sort_index()
full_df["y_true"] = y_train

In [23]:
full_df.dropna(inplace=True)

# 3.1 Find Tau (Volatility Threshold)

In [31]:
# As Provided in the paper
full_df["tau"] = np.quantile(full_df["y_true"], 0.8) # Threshold 

# 3.2 Regime Labelling 

In [32]:
full_df["regime"] = np.where(full_df["y_true"] >= full_df["tau"], "high", "low")

In [34]:
D_low = full_df[full_df["regime"] == "low"]
D_high = full_df[full_df["regime"] == "high"]

In [ ]:
# full_df.to_csv("../outputs/full_df.csv")

# 3.3 Demonstration Sampling

In [40]:
def random_sampling(demonstrations, K):
    return random.sample(demonstrations, K) # Uniform Sampling over len(demonstrations) for K elements. 

def sample_fixed_prior(Dlow, Dhigh, K, alpha=0.6):
    n_low = int(np.floor(K * alpha))
    n_high = K - n_low
    samples = []
    low_vol_samples = random.sample(Dlow, n_low)
    high_vol_samples = random.sample(Dhigh, n_high)
    samples = low_vol_samples + high_vol_samples
    random.shuffle(samples)
    return samples

def sample_estimated_label(Dlow, Dhigh, recent_vars, tau_prime, alpha_low, alpha_high, K, m):
    s_t = np.mean(recent_vars[-m: ])
    alpha = alpha_high if s_t >= tau_prime else alpha_low
    return sample_fixed_prior(Dlow, Dhigh, K, alpha)

def get_tau_prime(df: pd.DataFrame):
    rolling_variance_train = df["y_true"].rolling(3).mean().dropna()
    tau_prime = np.quantile(rolling_variance_train, 0.8)
    return tau_prime

In [42]:
get_tau_prime(full_df)

np.float64(0.00011298379091905507)